# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access dataset metadata as an object
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Dataset ID: {dataset.metadata.id}")

## 2. Data Overview
Review available record sets (`cr:RecordSet`), fields, columns, and their `@id`s.

We show all available record sets and their structure. Each element is referenced by its `@id`.

In [ ]:
# List available record sets and their fields as referenced by @id
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record Set: {rs.id}")
    if hasattr(rs, 'name'):
        print(f"  Name: {rs.name}")
    if hasattr(rs, 'description'):
        print(f"  Description: {rs.description}")
    # Each field in this RecordSet
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for f in rs.fields:
            print(f"    Field: {f.id}")
            if hasattr(f, 'name'):
                print(f"      Name: {f.name}")
            if hasattr(f, 'data_type'):
                print(f"      DataType: {f.data_type}")
            if hasattr(f, 'column') and f.column:
                if isinstance(f.column, list):
                    for c in f.column:
                        print(f"      Column: {c.id}")
                else:
                    print(f"      Column: {f.column.id}")
    print()

## 3. Data Extraction
Load all records from a chosen record set into a DataFrame for analysis. 

All extraction specifies `record_set` by its `@id`. We'll extract each and display one as a demonstration.

In [ ]:
# Collect all record set @id values
record_set_ids = [rs.id for rs in record_sets]
print("Record sets to extract:", record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set {record_set_id}")

# Choose the first record set for demonstration
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f"\nColumns for {example_record_set_id}:")
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, or grouping. All operations reference fields via their `@id`.

We'll select a numeric field and a possible categorical field from the first loaded record set.

In [ ]:
# For demonstration, pick a numeric field from the first record set
example_rs = record_sets[0] if record_sets else None

numeric_field_id = None
group_field_id = None

if example_rs is not None and hasattr(example_rs, 'fields'):
    # Find a numeric field
    for f in example_rs.fields:
        if hasattr(f, 'data_type') and f.data_type in ['Integer', 'Float', 'Number']:
            numeric_field_id = f.id
            break
    # Find a string/categorical field
    for f in example_rs.fields:
        if hasattr(f, 'data_type') and f.data_type == 'Text':
            group_field_id = f.id
            break

df = dataframes[example_record_set_id]

if numeric_field_id is None or numeric_field_id not in df.columns:
    print("No suitable numeric field found in this record set.")
else:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    norm_field = f"{numeric_field_id}_normalized"
    filtered_df[norm_field] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_field]].head())

    if group_field_id is not None and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (showing mean {numeric_field_id}):")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships in the dataset. All fields are referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Display a histogram and boxplot for the numeric field if available
if numeric_field_id is not None and numeric_field_id in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)

    plt.subplot(1, 2, 2)
    sns.boxplot(x=df[numeric_field_id].dropna())
    plt.title(f"Boxplot of {numeric_field_id}")
    plt.xlabel(numeric_field_id)

    plt.tight_layout()
    plt.show()

# Show barplot of mean value by group if possible
if group_field_id is not None and group_field_id in df.columns and numeric_field_id is not None and numeric_field_id in df.columns:
    gb = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    plt.figure(figsize=(10, 4))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=gb)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset contains record sets with various fields referenced by their unique `@id`s, which are critical for precise references and reproducibility.
- Example explorations included loading, filtering, normalizing, and visualizing data using only these unique identifiers.
- The structure and metadata, accessible through `mlcroissant`, make it straightforward to interact with complex FAIR datasets for data science analysis.

**Next steps:** Use the field and record set `@id`s for feature engineering, modeling, or in-depth analysis.